In [ ]:
import os
import pandas as pd
from datetime import datetime, timedelta, timezone
from icalendar import Calendar, Event
import hashlib
import pytz

In [ ]:
time_zone = "America/Toronto"

session = 'spring'
data_path = f'2026-27_FNML_{session}'
output_folder = f'{session}'
league_name = f'FNML 2026-27 — {session.title()}'
location = 'Bill Bolton Arena'
event_duration = 60 # In minutes

home_desc = 'Home - Yellow'
away_desc = 'Away — Blue'

playoff_descriptions = [
    'Final',
    'Semi-Final',
    'Quarter Final',
]

os.mkdir(output_folder, exist_ok=True)

In [25]:
tz = pytz.timezone(time_zone)

In [26]:
def string_to_uid(s):
    return hashlib.sha1(s.encode("utf-8")).hexdigest() + "@hockey"

def make_uid(row, team):
    raw = f"BBA-{row.Date}-{row.Time}-{team}"
    return string_to_uid(raw)

In [27]:
def longest_common_prefix(strings):
    strings = list(strings)
    if not strings:
        return ""

    # sort so extremes determine prefix
    strings.sort()

    first = strings[0]
    last = strings[-1]

    i = 0
    while i < len(first) and i < len(last) and first[i] == last[i]:
        i += 1

    return first[:i]

In [28]:
def load_csv_file(path):
    # Import
    df = pd.read_csv(path)

    # Date and time columns
    df['DateTime'] = pd.to_datetime(
        df['Date']+' '+df['Time'],
        format='%Y-%m-%d %I:%M%p',
        )
    df['Date'] = df['DateTime'].dt.date
    df['Time'] = df['DateTime'].dt.time

    # Split regular and playoff schedules
    regular_mask = df['Game'].str.contains('#')
    regular_df = df[regular_mask]
    playoff_df = df[~regular_mask]

    return regular_df, playoff_df

In [29]:
def get_teams(df):
    teams = pd.unique(df[['Home', 'Visitor']].values.ravel('K')).tolist()
    teams.sort()
    return teams

In [30]:
def process_regular_game(team, row, cal):
    # Find out time
    start_time = tz.localize(row.DateTime)
    end_time = start_time + timedelta(minutes=event_duration)

    # Extract game details
    at_home = row.Home == team
    opposite_team = row.Visitor if at_home else row.Home
    description = f"Game {row.Game} against {opposite_team} ({home_desc if at_home else away_desc})"

    # Create and populate event
    event = Event()
    event.add("uid", make_uid(row, team))
    event.add("dtstamp", datetime.now(timezone.utc))
    event.add("summary", f"{league_name}")
    event.add("description", description)
    event.add("dtstart", start_time)
    event.add("dtend", end_time)
    event.add("location", location)

    # Add event to calendar
    cal.add_component(event)

    return cal

In [31]:
def process_regular_games(team, df, cal):
    # Extract team schedule
    team_mask = (df['Home'] == team) | (df['Visitor'] == team)
    team_df = df[team_mask]

    # Parse row by row
    for row in team_df.itertuples(index=False):
        cal = process_regular_game(team, row, cal)
    
    return cal

In [32]:
def process_playoff_games(team, df, cal, descriptions=playoff_descriptions):
    for i, date in enumerate(sorted(df['Date'].unique(), reverse=True)):
        start_time = min(df[df['Date']==date]['DateTime'])
        start_time = tz.localize(start_time)

        num_games = sum(df['Date']==date)
        end_time = start_time + timedelta(minutes = num_games * event_duration)

        event_name = f'Playoff {descriptions[i]}'
        
        event = Event()
        event.add("uid", string_to_uid(f"BBA-{date}-{event_name}-{team}"))
        event.add("dtstamp", datetime.now(timezone.utc))
        event.add("summary", f"{league_name}")
        event.add("description", event_name)
        event.add("dtstart", start_time)
        event.add("dtend", end_time)
        event.add("location", location)

        cal.add_component(event)
    
    return cal

In [ ]:
def process_team(team, session, regular_df, playoff_df):
    cal = Calendar()
    cal.add("x-wr-calname", f"{league_name} Team {team} Schedule")
    cal.add("x-wr-timezone", time_zone)

    cal = process_regular_games(team, regular_df, cal)
    cal = process_playoff_games(team, playoff_df, cal)

    with open(f'{output_folder}/team_{team}.ics', 'wb') as f:
        f.write(cal.to_ical())

In [34]:
regular_df, playoff_df = load_csv_file(f'{data_path}.csv')
teams = get_teams(regular_df)
for team in teams:
    process_team(team, session, regular_df, playoff_df)